# Training deep models well: regularization and transfer learning

Machine Learning Engineer track, lesson MLE6. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0. The photos of unit 4 are from the Oxford-IIIT Pet dataset (Parkhi, Vedaldi, Zisserman and Jawahar), CC BY-SA 4.0; the copyright remains with the original owners of the images.

**Goals:** add dropout and early stopping to the training loop of lesson MLE5 and compare validation curves, see what batch size and learning rate change, and start from a pretrained model instead of from nothing, always against a baseline and one number.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The app-events data is synthetic: a script made it from stated rules. A CPU runtime is enough; unit 4 downloads about 800 MB of photos once.

## Setup

In [ ]:
import copy
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd
import torch
from torch import nn

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 2026       # the seed of the whole track
torch.set_num_threads(1)       # one thread: the sums are added in the same order on every run
DEVICE = "cpu"                 # these networks are small: a GPU would not make them faster

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The split, the scaled columns and the tensors

The split and the fitting and validation parts of the Data Scientist track (10,021 and 3,341 people; the test part is not scored in this lesson), the scaled columns of lesson MLE1 and the tensors of lesson MLE5. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)
prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), make_column_selector(dtype_exclude="number")),
                          ("num", StandardScaler(), make_column_selector(dtype_include="number"))]).fit(X_fit)


def tensors(rows, labels):
    """The scaled columns and the 0 or 1 labels as float tensors; the labels are a column of numbers."""
    return torch.tensor(prep.transform(rows), dtype=torch.float32), torch.tensor(labels.to_numpy(), dtype=torch.float32).unsqueeze(1)


Xf, yf = tensors(X_fit, y_fit)
Xv, yv = tensors(X_val, y_val)
logit = LogisticRegression(max_iter=1000).fit(prep.transform(X_fit), y_fit)       # the day-7 model, on the same columns
logit_chances = logit.predict_proba(prep.transform(X_val))[:, 1]
print(Xf.shape[0], Xv.shape[0], Xf.shape[1], round(float(roc_auc_score(y_val, logit_chances)), 3))

## The tools of this lesson

The pilot of lesson MLE5, 500 of the fitting people, the loss, and the training function of unit 3 of MLE5 with the lines this lesson explains marked `NEW`. Run this cell as it is. `val_auc` scores a model on the validation people with one number, and the logistic regression fitted on the same 500 people is the baseline.

In [ ]:
loss_fn = nn.BCELoss()                                                     # binary cross-entropy, as in lesson MLE5
pilot = X_fit.sample(500, random_state=SEED)                               # 500 of the fitting people
Xp, yp = tensors(pilot, y_fit.loc[pilot.index])

In [ ]:
def fit_network(Xa, ya, hidden, lr, epochs, batch=256, seed=SEED, dropout=0.0, weight_decay=0.0, patience=None):
    """Train a network with one hidden layer; return it and the loss on the fitting and validation people after each epoch."""
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(Xa.shape[1], hidden), nn.ReLU(),
                          nn.Dropout(dropout),                             # NEW: switches off a random share of the hidden outputs
                          nn.Linear(hidden, 1), nn.Sigmoid())
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)    # NEW: weight_decay is used in Exercise 1
    history, best, best_state, waited = [], float("inf"), None, 0
    for epoch in range(epochs):
        model.train()                                                      # NEW: training mode
        order = torch.randperm(len(Xa))
        for start in range(0, len(Xa), batch):
            rows = order[start:start + batch]
            optimizer.zero_grad()
            loss_fn(model(Xa[rows]), ya[rows]).backward()
            optimizer.step()
        model.eval()                                                       # NEW: evaluation mode, to measure the loss
        with torch.no_grad():
            history.append((float(loss_fn(model(Xa), ya)), float(loss_fn(model(Xv), yv))))
        if history[-1][1] < best:                                          # NEW: remember the numbers of the lowest validation loss so far
            best, best_state, waited = history[-1][1], copy.deepcopy(model.state_dict()), 0
        else:
            waited += 1
            if patience is not None and waited >= patience:                # NEW: stop after `patience` epochs without a new lowest
                break
    if patience is not None:
        model.load_state_dict(best_state)                                  # NEW: give back the numbers of the lowest point
    return model, history

In [ ]:
def val_auc(model):
    """ROC AUC of a model on the validation people, rounded to 3 decimals."""
    model.eval()
    with torch.no_grad():
        return round(float(roc_auc_score(y_val, model(Xv).squeeze(1).numpy())), 3)


logit_pilot = LogisticRegression(max_iter=1000).fit(prep.transform(pilot), y_fit.loc[pilot.index])      # the baseline: same 500 people
print(Xp.shape[0], round(float(roc_auc_score(y_val, logit_pilot.predict_proba(prep.transform(X_val))[:, 1])), 3))

## Unit 1: Dropout

**Predict first.** The 64-neuron network of lesson MLE5 learns from the same 500 people for 300 epochs, once with no dropout and once with half of the hidden outputs switched off at every step (`dropout=0.5`). Will dropout lower the lowest validation loss of the 300 epochs, the validation loss at epoch 300, or both?

In [ ]:
for p in [0.0, 0.5]:
    model, history = fit_network(Xp, yp, 64, 0.1, 300, batch=50, dropout=p)
    validation = [pair[1] for pair in history]
    print(p, round(min(validation), 3), round(history[-1][0], 3), round(history[-1][1], 3))

A check on the headline: 20 other pilots (optional, about a minute). Each is 500 other fitting people; the lesson quotes what it prints.

In [ ]:
import statistics

rows = []
for seed in range(100, 120):
    pilot_k = X_fit.sample(500, random_state=seed)
    Xk, yk = tensors(pilot_k, y_fit.loc[pilot_k.index])
    row = {}
    for p in [0.0, 0.5]:
        _, history_k = fit_network(Xk, yk, 64, 0.1, 300, batch=50, dropout=p)
        row[p] = (min(pair[1] for pair in history_k), history_k[-1][1])
    stopped, _ = fit_network(Xk, yk, 64, 0.1, 300, batch=50, patience=10)
    baseline = LogisticRegression(max_iter=1000).fit(prep.transform(pilot_k), y_fit.loc[pilot_k.index])
    row["stopped"] = val_auc(stopped)
    row["baseline"] = round(float(roc_auc_score(y_val, baseline.predict_proba(prep.transform(X_val))[:, 1])), 3)
    rows.append(row)
for p in [0.0, 0.5]:
    print(p, "lowest", round(statistics.median(r[p][0] for r in rows), 3), "at epoch 300", round(statistics.median(r[p][1] for r in rows), 3),
          "range", round(min(r[p][1] for r in rows), 3), round(max(r[p][1] for r in rows), 3))
print("dropout lower at epoch 300 in", sum(r[0.5][1] < r[0.0][1] for r in rows), "of", len(rows))
print("stopped AUC median", round(statistics.median(r["stopped"] for r in rows), 3), "baseline median", round(statistics.median(r["baseline"] for r in rows), 3),
      "stopped at least as high in", sum(r["stopped"] >= r["baseline"] for r in rows), "of", len(rows))

## Unit 2: Early stopping

**Predict first.** The same network, no dropout, is trained with `patience=10` and, as in unit 1, for all 300 epochs. A logistic regression fitted on the same 500 people is the baseline. Which has the higher validation AUC, the network that stopped early or the one that trained for 300 epochs, and how does the stopped network compare with the baseline?

In [ ]:
model, history = fit_network(Xp, yp, 64, 0.1, 300, batch=50, patience=10)
best_epoch = int(np.argmin([pair[1] for pair in history])) + 1
long_model, _ = fit_network(Xp, yp, 64, 0.1, 300, batch=50)                    # the same network for all 300 epochs
print(len(history), best_epoch, val_auc(model), val_auc(long_model))
print(round(float(roc_auc_score(y_val, logit_pilot.predict_proba(prep.transform(X_val))[:, 1])), 3))

## Unit 3: Batch size and learning rate

**Predict first.** A network with 8 hidden neurons learns from all 10,021 fitting people for 10 epochs, in six runs: batches of 32, 256 and 1,024 people, each with learning rate 0.1 and 1.0. Which run ends with the lowest validation loss, and does one learning rate win at every batch size?

In [ ]:
for batch in [32, 256, 1024]:
    for lr in [0.1, 1.0]:
        model, history = fit_network(Xf, yf, 8, lr, 10, batch=batch)
        updates = -(-len(Xf) // batch) * 10                                    # steps taken: batches per epoch times 10 epochs
        print(batch, lr, updates, round(history[-1][1], 3))

## The photos (given cells for unit 4)

Six dog and cat breeds of the Oxford-IIIT Pet dataset, the first six in its alphabetical list, fixed before any model ran: 10 photos of each to learn from (the first 10 of the `trainval` list) and 30 of each to test (the first 30 of the `test` list), 60 and 180 photos. The first cell downloads the dataset once (about 800 MB, about a minute) and turns each photo into the 3 by 224 by 224 numbers that ResNet-18 expects. Run it as it is. The flat guess for six equal breeds is 1 in 6.

In [ ]:
from torchvision import datasets, models

PETS_ROOT = "pets"                                                            # the dataset is downloaded here the first time
WEIGHTS = models.ResNet18_Weights.IMAGENET1K_V1                               # the numbers of the pretrained network
prepare = WEIGHTS.transforms()                                                # resize, crop, scale and normalize, as the network was trained
pets_fit = datasets.OxfordIIITPet(PETS_ROOT, split="trainval", target_types="category", transform=prepare, download=True)
pets_test = datasets.OxfordIIITPet(PETS_ROOT, split="test", target_types="category", transform=prepare, download=True)
BREEDS = pets_fit.classes[:6]


def pick(dataset, per_breed):
    """The first `per_breed` photos of each of the six breeds, and their breed numbers."""
    chosen = [i for breed in range(6) for i in [j for j, number in enumerate(dataset._labels) if number == breed][:per_breed]]
    return torch.stack([dataset[i][0] for i in chosen]), torch.tensor([dataset._labels[i] for i in chosen])


images_fit, labels_fit = pick(pets_fit, 10)
images_test, labels_test = pick(pets_test, 30)
print(BREEDS)
print(images_fit.shape, images_test.shape, round(1 / 6, 3))

A small convolutional network that learns from nothing, and the function that trains any classifier on photos or on numbers. Run this cell as it is. `Scratch` has about 24,000 numbers and starts from random values, like the networks of lesson MLE5.

In [ ]:
class Scratch(nn.Module):
    """A small network that reads pixels: three layers that slide over the photo, then one output number for each breed."""

    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(3, 16, 5, 2, 2), nn.ReLU(), nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(), nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
                                    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 6))

    def forward(self, x):
        return self.layers(nn.functional.interpolate(x, size=112))              # a smaller photo keeps this cell fast


def fit_classifier(model, x, y, epochs, lr, batch=None):
    """Adam on the cross-entropy loss; every epoch uses all of x, or batches of `batch` photos."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x))
        for start in range(0, len(x), batch or len(x)):
            rows = order[start:start + (batch or len(x))]
            optimizer.zero_grad()
            nn.functional.cross_entropy(model(x[rows]), y[rows]).backward()
            optimizer.step()


def accuracy(model, x, y):
    """The share of photos whose highest output is the right breed."""
    model.eval()
    with torch.no_grad():
        return round(float((model(x).argmax(1) == y).float().mean()), 3)


print(sum(p.numel() for p in Scratch().parameters()))

## Unit 4: A pretrained start

**Predict first.** Two networks learn the 60 photos. One is `Scratch`, trained from random numbers on the pixels. The other is ResNet-18 with its numbers from ImageNet and a new last layer, the only part that learns. Each trains for 200 epochs. On the 180 test photos, which scores higher and by how much, and does either one score 1.0 on the 60 photos it learned from?

In [ ]:
backbone = models.resnet18(weights=WEIGHTS)                                # ResNet-18 with the numbers learned on ImageNet
for parameter in backbone.parameters():
    parameter.requires_grad = False                                        # freeze: these numbers stay as they are
backbone.fc = nn.Identity()                                                # take off its last layer: 512 numbers come out for each photo
backbone.eval()
with torch.no_grad():
    features_fit, features_test = backbone(images_fit), backbone(images_test)      # computed once, because nothing in the backbone learns
torch.manual_seed(SEED)
head = nn.Linear(512, 6)                                                   # the new last layer: the only numbers that learn
print(sum(p.numel() for p in backbone.parameters()), sum(p.numel() for p in head.parameters()))
fit_classifier(head, features_fit, labels_fit, 200, 0.01)
torch.manual_seed(SEED)
scratch = Scratch()
fit_classifier(scratch, images_fit, labels_fit, 200, 0.003, batch=20)
print("pretrained, new last layer", accuracy(head, features_test, labels_test), accuracy(head, features_fit, labels_fit))
print("from nothing", accuracy(scratch, images_test, labels_test), accuracy(scratch, images_fit, labels_fit))

A picture of the photos and of the test photos that the pretrained network gets wrong, for Colab. Run this cell as it is.

In [ ]:
import matplotlib.pyplot as plt

wrong = (head(features_test).argmax(1) != labels_test).nonzero().squeeze(1).tolist()
raw_test = datasets.OxfordIIITPet(PETS_ROOT, split="test", target_types="category")
raw_fit = datasets.OxfordIIITPet(PETS_ROOT, split="trainval", target_types="category")
first_test = [i for breed in range(6) for i in [j for j, number in enumerate(raw_test._labels) if number == breed][:30]]
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for breed in range(6):
    first = [j for j, number in enumerate(raw_fit._labels) if number == breed][0]
    axes[0][breed].imshow(raw_fit[first][0])
    axes[0][breed].set_title(BREEDS[breed], fontsize=8)
for slot, i in enumerate(wrong[:6]):
    axes[1][slot].imshow(raw_test[first_test[i]][0])
    axes[1][slot].set_title(f"{BREEDS[int(labels_test[i])]} -> {BREEDS[int(head(features_test[i:i + 1]).argmax(1))]}", fontsize=7)
for axis in axes.ravel():
    axis.axis("off")
plt.tight_layout()
plt.show()
print(len(wrong), np.bincount(labels_test[wrong].numpy(), minlength=6).tolist())                # wrong photos in all, and for each breed

A check on the headline: 20 other draws of 10 training photos for each breed (optional, a few minutes). Each draw is another random 10 of the `trainval` photos of the six breeds; the test photos stay the same.

In [ ]:
rng = np.random.default_rng(SEED)
fit_pool = [[j for j, number in enumerate(pets_fit._labels) if number == breed] for breed in range(6)]
pool_ids = sorted(i for ids in fit_pool for i in ids)
pool_images = torch.stack([pets_fit[i][0] for i in pool_ids])
with torch.no_grad():
    pool_features = torch.cat([backbone(pool_images[s:s + 50]) for s in range(0, len(pool_images), 50)])
slot = {i: k for k, i in enumerate(pool_ids)}
head_scores, scratch_scores = [], []
for draw in range(20):
    rng = np.random.default_rng(100 + draw)
    chosen = [slot[int(i)] for ids in fit_pool for i in rng.choice(ids, 10, replace=False)]
    y_draw = torch.tensor([pets_fit._labels[pool_ids[k]] for k in chosen])
    torch.manual_seed(SEED)
    head_d = nn.Linear(512, 6)
    fit_classifier(head_d, pool_features[chosen], y_draw, 200, 0.01)
    head_scores.append(accuracy(head_d, features_test, labels_test))
    torch.manual_seed(SEED)
    scratch_d = Scratch()
    fit_classifier(scratch_d, pool_images[chosen], y_draw, 200, 0.003, batch=20)
    scratch_scores.append(accuracy(scratch_d, images_test, labels_test))
print("pretrained", sorted(head_scores)[10], min(head_scores), max(head_scores))
print("from nothing", sorted(scratch_scores)[10], min(scratch_scores), max(scratch_scores))

## Finish it

### Exercise 1: Finish it

**Task:** **Weight decay.** `fit_network` has one more argument marked `NEW`, `weight_decay`. Train the 64-neuron network on the 500 people, for 300 epochs with batches of 50, as in unit 1 but with `weight_decay=0.01` and no dropout, and put the validation loss at epoch 300, rounded to 3 decimals, in `answer_1`.

**Expected:** `answer_1` is a number between 0.6 and 0.8.

In [ ]:
model, history = fit_network(Xp, yp, 64, 0.1, 300, batch=50, weight_decay=0.0)     # given: change the weight decay to 0.01
answer_1 = None  # replace None with the validation loss after the last epoch, rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", 0.655, traps={0.874: "That is the loss with no weight decay: the call still has weight_decay=0.0."}, tol=5e-4)

<details>
<summary>Hint</summary>

Each value of `history` is a pair: the loss on the 500 people and the loss on the validation people. You want the second number of the last pair.

</details>

<details>
<summary>Half-solution</summary>

```python
model, history = fit_network(Xp, yp, 64, 0.1, 300, batch=50, weight_decay=0.01)
answer_1 = round(history[-1][1], 3)
```

</details>

<details>
<summary>Solution</summary>

```python
model, history = fit_network(Xp, yp, 64, 0.1, 300, batch=50, weight_decay=0.01)
answer_1 = round(history[-1][1], 3)
```

0.655. With `weight_decay=0.01` the update adds 0.01 times each number to its slope (the formula on the SGD page, where the page calls it a weight decay, an L2 penalty), so every step pulls every number a little toward 0. That is a third way to make memorizing harder: the validation loss at epoch 300 is 0.655, below the 0.717 of dropout 0.5 and the 0.874 of no regularization in unit 1.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Dropout and patience from a blank cell.** Before you start, say in one sentence what you expect: with `dropout=0.7` and `patience=10`, does the network keep an earlier or a later epoch than the 28 of unit 2? Then write the cell yourself, in the empty cell below: train the 64-neuron network on the 500 people with batches of 50, learning rate 0.1, up to 300 epochs, `dropout=0.7` and `patience=10`. Put the epoch whose numbers it keeps and the validation AUC of the kept network, rounded to 3 decimals, in a list `answer_2`.

**Expected:** `answer_2` is a list of two numbers, a whole epoch number and an AUC a little below 0.7.

In [ ]:
_ = check("Exercise 2", "answer_2", [35, 0.683], traps={(65, 0.655): "That is the last epoch's AUC of a run without patience: the run has to stop early and keep the best numbers."}, tol=5e-4)

<details>
<summary>Hint</summary>

The function returns the model and the history. The model already holds the kept numbers, so `val_auc(model)` scores them. The kept epoch is where the validation loss is lowest, or `len(history) - 10`, because the run stops 10 epochs after it.

</details>

<details>
<summary>Half-solution</summary>

```python
model, history = fit_network(Xp, yp, 64, 0.1, 300, batch=50, dropout=0.7, patience=10)
...
```

</details>

<details>
<summary>Solution</summary>

```python
model, history = fit_network(Xp, yp, 64, 0.1, 300, batch=50, dropout=0.7, patience=10)
kept_epoch = int(np.argmin([pair[1] for pair in history])) + 1
answer_2 = [kept_epoch, val_auc(model)]
```

[35, 0.683]: the run stopped after 45 epochs and kept epoch 35, later than the 28 of unit 2, with about the same AUC (0.683 against 0.681). Dropout makes the validation curve fall more slowly, so its lowest point comes later; `patience=10` waited for it and `load_state_dict` gave its numbers back.

</details>

## Check yourself

### Exercise 3: Check yourself

**Task:** **Spot the flaw.** A colleague copies `fit_network` for a network with `dropout=0.4`, but writes the loop without `model.train()` and `model.eval()`. Change the function so that the loss is measured the way unit 1 measures it, and put the validation loss at epoch 300 (500 people, 64 neurons, batches of 50, learning rate 0.1), rounded to 3 decimals, in `answer_3`.

**Expected:** `answer_3` is a number between 0.6 and 0.8.

In [ ]:
def fit_unswitched(Xa, ya, hidden, lr, epochs, batch=256, seed=SEED, dropout=0.0):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(Xa.shape[1], hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1), nn.Sigmoid())
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        order = torch.randperm(len(Xa))
        for start in range(0, len(Xa), batch):
            rows = order[start:start + batch]
            optimizer.zero_grad()
            loss_fn(model(Xa[rows]), ya[rows]).backward()
            optimizer.step()
        with torch.no_grad():
            history.append((float(loss_fn(model(Xa), ya)), float(loss_fn(model(Xv), yv))))
    return history


answer_3 = round(fit_unswitched(Xp, yp, 64, 0.1, 300, batch=50, dropout=0.4)[-1][1], 3)
# change the function so that the loss is measured the way unit 1 measures it

In [ ]:
_ = check("Exercise 3", "answer_3", 0.723, traps={0.77: "That is the loss measured with the dropout still on: the loop never switches the network to evaluation mode."}, tol=5e-4)

<details>
<summary>Hint</summary>

What state is the network in when the loss is measured? Where does `fit_network` change it, and where does it change it back?

</details>

<details>
<summary>Half-solution</summary>

```python
    for epoch in range(epochs):
        model.train()
        ...
        model.eval()
        with torch.no_grad():
            history.append(...)
```

</details>

<details>
<summary>Solution</summary>

```python
def fit_unswitched(Xa, ya, hidden, lr, epochs, batch=256, seed=SEED, dropout=0.0):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(Xa.shape[1], hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1), nn.Sigmoid())
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(Xa))
        for start in range(0, len(Xa), batch):
            rows = order[start:start + batch]
            optimizer.zero_grad()
            loss_fn(model(Xa[rows]), ya[rows]).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            history.append((float(loss_fn(model(Xa), ya)), float(loss_fn(model(Xv), yv))))
    return history


answer_3 = round(fit_unswitched(Xp, yp, 64, 0.1, 300, batch=50, dropout=0.4)[-1][1], 3)
```

0.723, the validation loss at epoch 300 when the network is switched to evaluation mode for the measurement. The flawed function reports 0.770: the network is still in training mode when it measures, so `Dropout` keeps switching off 40 percent of the hidden outputs while the loss is computed. The PyTorch page says that during evaluation the module computes an identity function, and the `Module.eval` page that `eval()` affects only certain modules, such as `Dropout` and `BatchNorm`.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")